#Dependencies

In [ ]:
!pip install datamapplot

In [ ]:
!pip install bertopic

#Importing Libraries

In [ ]:
import os
import re
import glob
import json
import zipfile
import random
import openai
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn import metrics
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from google.colab import files
from umap import UMAP
import umap
from hdbscan import HDBSCAN
import hdbscan
from bertopic import BERTopic
import bertopic
from bertopic.representation import OpenAI
import datamapplot
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration
import plotly.express as px
import plotly.io as pio

#Downloading Dataset from Kaggle

In [ ]:
!curl -L -o /content/arxiv.zip https://www.kaggle.com/api/v1/datasets/download/Cornell-University/arxiv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1515M  100 1515M    0     0   204M      0  0:00:07  0:00:07 --:--:--  224M


In [ ]:
!unzip "/content/arxiv.zip"

Archive:  /content/arxiv.zip
replace arxiv-metadata-oai-snapshot.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
!rm "/content/arxiv.zip"

In [ ]:
total_papers = 0
with open('/content/arxiv-metadata-oai-snapshot.json', 'r') as f:
    for _ in tqdm(f, desc="Counting papers"):
        total_papers += 1

Counting papers: 2803128it [00:03, 756480.47it/s]


In [ ]:
2735264

2735264

In [ ]:
chunks = []
chunk_size = 10000
paper_count = 0


In [ ]:
with open('/content/arxiv-metadata-oai-snapshot.json', 'r') as f:
    for i, line in enumerate(tqdm(f, total=total_papers)):
        paper = json.loads(line)
        chunks.append({
            'id': paper['id'],
            'abstract': paper.get('abstract', ''),
            'categories': paper.get('categories', '').split(),
            'year': paper.get('update_date', '')[:4]
        })
        paper_count += 1

        # Save chunks periodically to avoid memory issues
        if (i + 1) % chunk_size == 0 and chunks:
            pd.DataFrame(chunks).to_parquet(f'chunk_{i}.parquet')
            chunks = []

# Save any remaining papers in the final chunk
if chunks:
    pd.DataFrame(chunks).to_parquet(f'chunk_{i}_final.parquet')

100%|██████████| 2803128/2803128 [01:08<00:00, 41022.26it/s]


In [ ]:
# Merge chunks (if multiple were created)
df = pd.concat([pd.read_parquet(f) for f in glob.glob('chunk_*.parquet')])

In [ ]:
df

,id,abstract,categories,year
0,1709.05134,This work is a pedagogical review dedicated ...,"[gr-qc, hep-th]",2019
1,1709.05135,The determinantal point process (DPP) is an ...,[cs.IR],2018
2,1709.05136,The Arcminute Microkelvin Imager (AMI) carri...,[astro-ph.GA],2017
3,1709.05137,We consider a random walk in dimension $d\ge...,[math.PR],2018
4,1709.05138,"Nano- and microscale particles, such as coll...","[cond-mat.soft, cond-mat.stat-mech, math-ph, m...",2017
...,...,...,...,...
9995,2209.14280,Ongoing progress in laser and accelerator te...,"[physics.plasm-ph, physics.acc-ph]",2023
9996,2209.14281,Multilingual search can be achieved with sub...,"[cs.CL, cs.AI, cs.LG]",2022
9997,2209.14282,"More than 65 years ago, John Wheeler suggest...","[gr-qc, hep-th]",2023
9998,2209.14283,Methods to identify cause-effect relationshi...,[stat.ME],2022


In [ ]:
df.columns

Index(['id', 'abstract', 'categories', 'year'], dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2803128 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   id          object
 1   abstract    object
 2   categories  object
 3   year        object
dtypes: object(4)
memory usage: 106.9+ MB


In [ ]:
df['categories']

,categories
0,"[gr-qc, hep-th]"
1,[cs.IR]
2,[astro-ph.GA]
3,[math.PR]
4,"[cond-mat.soft, cond-mat.stat-mech, math-ph, m..."
...,...
9995,"[physics.plasm-ph, physics.acc-ph]"
9996,"[cs.CL, cs.AI, cs.LG]"
9997,"[gr-qc, hep-th]"
9998,[stat.ME]


# Data Preparation

In [ ]:
def extract_id(paper_id):
    # For old format: strip subject prefix
    if '/' in paper_id:
        return paper_id.split('/')[-1]
    # For new/modern: keep as-is
    return paper_id

df['id'] = df['id'].apply(extract_id)

In [ ]:
df

,id,abstract,categories,year
0,1709.05134,This work is a pedagogical review dedicated ...,"[gr-qc, hep-th]",2019
1,1709.05135,The determinantal point process (DPP) is an ...,[cs.IR],2018
2,1709.05136,The Arcminute Microkelvin Imager (AMI) carri...,[astro-ph.GA],2017
3,1709.05137,We consider a random walk in dimension $d\ge...,[math.PR],2018
4,1709.05138,"Nano- and microscale particles, such as coll...","[cond-mat.soft, cond-mat.stat-mech, math-ph, m...",2017
...,...,...,...,...
9995,2209.14280,Ongoing progress in laser and accelerator te...,"[physics.plasm-ph, physics.acc-ph]",2023
9996,2209.14281,Multilingual search can be achieved with sub...,"[cs.CL, cs.AI, cs.LG]",2022
9997,2209.14282,"More than 65 years ago, John Wheeler suggest...","[gr-qc, hep-th]",2023
9998,2209.14283,Methods to identify cause-effect relationshi...,[stat.ME],2022


#Abstract Cleaning

In [ ]:
def clean_abstract(text):
    """
    Clean abstracts for embedding/clustering:
    1. Lowercase
    2. Replace LaTeX symbols (e.g., \ell_0 → L0, \alpha_1 → alpha1)
    3. Remove LaTeX equations (e.g., \frac{1}{2})
    4. Remove URLs, arXiv IDs, emails
    5. Keep letters, numbers, basic punctuation
    6. Remove extra whitespace
    """
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Step 1: Replace LaTeX symbols with subscripts (e.g., \ell_0 → L0)
    latex_symbols = {
        r'\\ell': 'L',       # \ell → L
        r'\\alpha': 'alpha',
        r'\\beta': 'beta',
        r'\\gamma': 'gamma',
        r'\\theta': 'theta',
        r'\\lambda': 'lambda',
        r'\\sigma': 'sigma',
        r'\\mu': 'mu',
        r'\\epsilon': 'epsilon'
    }

    # Handle subscripts (e.g., \ell_0 → L0, \alpha_{1} → alpha1)
    for pattern, replacement in latex_symbols.items():
        # Case 1: \symbol_{number} → symbolnumber (e.g., \alpha_{1} → alpha1)
        text = re.sub(pattern + r'_\{(\d+)\}', replacement + r'\1', text)
        # Case 2: \symbol_number → symbolnumber (e.g., \alpha_1 → alpha1)
        text = re.sub(pattern + r'_(\d+)', replacement + r'\1', text)
        # Case 3: Standalone symbols (e.g., \alpha → alpha)
        text = re.sub(pattern, replacement, text)

    # Step 2: Remove remaining LaTeX commands (e.g., \frac{1}{2})
    text = re.sub(r'\\[a-zA-Z]+\{.*?\}', ' ', text)

    # Step 3: Remove URLs, arXiv IDs, emails
    text = re.sub(r'http\S+|www\S+|arxiv:\d+\.\d+|@\S+', ' ', text)

    # Step 4: Keep letters, numbers, and basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s.,;:!?\'"-]', ' ', text)

    # Step 5: Collapse whitespace and trim
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
tqdm.pandas(desc="Cleaning abstracts")
df['cleaned_abstract'] = df['abstract'].progress_apply(clean_abstract)

Cleaning abstracts: 100%|██████████| 2803128/2803128 [05:58<00:00, 7828.04it/s]


In [ ]:
df

,id,abstract,categories,year,cleaned_abstract
0,1709.05134,This work is a pedagogical review dedicated ...,"[gr-qc, hep-th]",2019,this work is a pedagogical review dedicated to...
1,1709.05135,The determinantal point process (DPP) is an ...,[cs.IR],2018,the determinantal point process dpp is an eleg...
2,1709.05136,The Arcminute Microkelvin Imager (AMI) carri...,[astro-ph.GA],2017,the arcminute microkelvin imager ami carried o...
3,1709.05137,We consider a random walk in dimension $d\ge...,[math.PR],2018,we consider a random walk in dimension d geq 1...
4,1709.05138,"Nano- and microscale particles, such as coll...","[cond-mat.soft, cond-mat.stat-mech, math-ph, m...",2017,"nano- and microscale particles, such as colloi..."
...,...,...,...,...,...
9995,2209.14280,Ongoing progress in laser and accelerator te...,"[physics.plasm-ph, physics.acc-ph]",2023,ongoing progress in laser and accelerator tech...
9996,2209.14281,Multilingual search can be achieved with sub...,"[cs.CL, cs.AI, cs.LG]",2022,multilingual search can be achieved with subwo...
9997,2209.14282,"More than 65 years ago, John Wheeler suggest...","[gr-qc, hep-th]",2023,"more than 65 years ago, john wheeler suggested..."
9998,2209.14283,Methods to identify cause-effect relationshi...,[stat.ME],2022,methods to identify cause-effect relationships...


In [ ]:
# Assuming df['categories'] contains lists of strings
unique_categories = set(cat for row in df['categories'] for cat in row)

# Optional: convert to sorted list if needed
unique_categories = sorted(unique_categories)
print(unique_categories)

['acc-phys', 'adap-org', 'alg-geom', 'ao-sci', 'astro-ph', 'astro-ph.CO', 'astro-ph.EP', 'astro-ph.GA', 'astro-ph.HE', 'astro-ph.IM', 'astro-ph.SR', 'atom-ph', 'bayes-an', 'chao-dyn', 'chem-ph', 'cmp-lg', 'comp-gas', 'cond-mat', 'cond-mat.dis-nn', 'cond-mat.mes-hall', 'cond-mat.mtrl-sci', 'cond-mat.other', 'cond-mat.quant-gas', 'cond-mat.soft', 'cond-mat.stat-mech', 'cond-mat.str-el', 'cond-mat.supr-con', 'cs.AI', 'cs.AR', 'cs.CC', 'cs.CE', 'cs.CG', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DB', 'cs.DC', 'cs.DL', 'cs.DM', 'cs.DS', 'cs.ET', 'cs.FL', 'cs.GL', 'cs.GR', 'cs.GT', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG', 'cs.LO', 'cs.MA', 'cs.MM', 'cs.MS', 'cs.NA', 'cs.NE', 'cs.NI', 'cs.OH', 'cs.OS', 'cs.PF', 'cs.PL', 'cs.RO', 'cs.SC', 'cs.SD', 'cs.SE', 'cs.SI', 'cs.SY', 'dg-ga', 'econ.EM', 'econ.GN', 'econ.TH', 'eess.AS', 'eess.IV', 'eess.SP', 'eess.SY', 'funct-an', 'gr-qc', 'hep-ex', 'hep-lat', 'hep-ph', 'hep-th', 'math-ph', 'math.AC', 'math.AG', 'math.AP', 'math.AT', 'math.CA', 'math.CO', 'math.

In [ ]:
total_unique = len(unique_categories)

print("Total unique categories:", total_unique)

Total unique categories: 176


In [ ]:
# Load your cleaned abstracts
documents = df['cleaned_abstract'].tolist()

In [ ]:
len(documents)

2803128

#Embedding Model Training from Hugging face

In [ ]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("BAAI/bge-small-en")
embeddings = embedding_model.encode(documents, show_progress_bar=True)

Batches:   0%|          | 0/87598 [00:00<?, ?it/s]

###Saving Embedding model

In [ ]:
# Save the SentenceTransformer embedding model
output_path = "./my_embedding_model"
embedding_model.save(output_path)

In [ ]:
# Using zipfile to compress the folder first (recommended for larger folders)
folder_path = './my_embedding_model'  # Path to the folder you want to download
zip_path = './my_embedding_model.zip'  # Path where the zip file will be created

# Create a zip file containing the folder
def zipdir(path, ziph):
    # Iterate through all files in the directory
    for root, dirs, files in os.walk(path):
        for file in files:
            # Create the full file path
            file_path = os.path.join(root, file)
            # Add the file to the zip archive with a relative path
            ziph.write(file_path, os.path.relpath(file_path, os.path.join(path, '..')))

# Create the zip file
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(folder_path, zipf)

# Download the zip file
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!unzip "/content/my_embedding_model.zip"

In [ ]:
embedding_model = SentenceTransformer('/content/my_embedding_model')  # Point to the unzipped folder


#Trying out UMAP

In [ ]:
#Reduced embeddings for visualization purposes
reduced_embeddings = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42).fit_transform(embeddings)

In [ ]:
df_plot = pd.DataFrame({
    "x1": [point[0] for point in reduced_embeddings],
    "x2": [point[1] for point in reduced_embeddings ],
    "documents": documents,
})

In [ ]:
df_plot.head()


In [ ]:
df_plot["docs_short"] = df_plot["documents"].str[:100] + "..."
df_plot.head(10)

In [ ]:
pio.renderers.default = 'colab'

total_docs = len(df_plot)
fig = px.scatter(df_plot, x="x1", y="x2",  hover_data=["docs_short"])
fig.update_traces(marker=dict(line=dict(width=0.5, color='white')))
fig.update_layout(
    title=f"Arxiv Abstract - Document Map (2.7M documents)",
    title_font_size=20
)

fig.show()

###UMAP & HDBscan Model Initialization

In [ ]:
umap_model = UMAP(n_neighbors=30, n_components=10, min_dist=0.1, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=100, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

###GPT 4o mini Initialization

In [ ]:

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <topic label>
"""

# Create your representation model
client = openai.OpenAI(api_key="xxxxxxxxxxxxx")
representation_model = OpenAI(client, prompt=prompt, model="gpt-4o-mini", delay_in_seconds=30)



### Topic Representation Models

In [ ]:

# KeyBERT
keybert = KeyBERTInspired()

# MMR
mmr = MaximalMarginalRelevance(diversity=0.3)

# All representation models
representation_model = {
    "KeyBERT": keybert,
    "MMR": mmr,
    "GPT-40": representation_model,
}

In [ ]:

topic_model = BERTopic(

  # Sub-models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  representation_model=representation_model,
  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(documents, embeddings)

NameError: name 'embeddings' is not defined

#Saving Bertopic Model

In [ ]:
topic_model.save("my_topic_model", serialization="safetensors", save_embedding_model=False)


In [ ]:
# Show topics
topic_model.get_topic_info()

#Model Visualization

In [ ]:
topic_model.visualize_topics()

In [ ]:
gpt4o_labels = [label[0][0].split("\n")[0] for label in topic_model.get_topics(full=True)["GPT-40"].values()]

In [ ]:
gpt4o_labels

In [ ]:
# Get document info
document_info = topic_model.get_document_info(documents)
document_info["GPT-40"] = document_info["GPT-40"].apply(lambda x: x[0])

In [ ]:
all_labels = document_info["GPT-40"]

In [ ]:
all_labels.unique()

In [ ]:
all_labels.nunique()

In [ ]:
topic_model.visualize_barchart()


In [ ]:
df_plot = pd.DataFrame({
    "x1": [point[0] for point in reduced_embeddings],
    "x2": [point[1] for point in reduced_embeddings],
    "docs": documents,
    "label": all_labels
})
df_plot["docs_short"] = df_plot["docs"].str[:100] + "..."
df_plot.head(10)

In [ ]:
import plotly.express as px
fig = px.scatter(df_plot, x="x1", y="x2", color="label", hover_data=["docs_short"])
fig.show()

In [ ]:
# Run the visualization
datamapplot.create_plot(
    reduced_embeddings,
    all_labels,

    use_medoids=True,
    figsize=(12, 12),
    dpi=100,

    title="Topic Analysis- Arxiv Abstracts",
    sub_title_keywords={
        "fontsize":18,
    },

    highlight_labels=[
        "Retinopathy Prematurity Screening",
    ],
    highlight_label_keywords={
        "fontsize": 12,
        "fontweight": "bold",
        "bbox": {"boxstyle":"round"}
    },
    label_font_size=8,
    label_wrap_width=16,
    label_linespacing=1.25,
    label_direction_bias=1.3,
    label_margin_factor=2.0,
    label_base_radius=15.0,
    point_size=4,
    marker_type="o",
    arrowprops={
        "arrowstyle":"wedge,tail_width=0.5",
        "connectionstyle":"arc3,rad=0.05",
        "linewidth":0,
        "fc":"#33333377"
    },

    add_glow=True,
    glow_keywords={
        "kernel_bandwidth": 0.75,  # controls how wide the glow spreads.
        "kernel": "cosine",        # controls the kernel type. Default is "gaussian". See https://scikit-learn.org/stable/modules/density.html#kernel-density.
        "n_levels": 32,            # controls how many "levels" there are in the contour plot.
        "max_alpha": 0.9,          # controls the translucency of the glow.
    },

    darkmode=False,
)
plt.tight_layout()

# Save the plot as a png
plt.savefig('plot_datamapplot.png')


#Model Evaluation

In [ ]:
# Basic model information
topic_info = topic_model.get_topic_info()
print(f"Number of topics found: {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents per topic (avg): {topic_info[topic_info['Topic'] != -1]['Count'].mean():.1f}")

# Topic diversity - check how unique the top words are across topics
all_top_words = []
unique_words = set()
for topic in topic_info[topic_info['Topic'] != -1]['Topic']:
    words = [word for word, _ in topic_model.get_topic(topic)]
    all_top_words.extend(words)
    unique_words.update(words)

diversity = len(unique_words) / len(all_top_words) if all_top_words else 0
print(f"Topic diversity score: {diversity:.4f} (higher is better)")

# Check if c_tf_idf is available and has the expected format
try:
    # Try to access the representation attribute directly
    coherence_values = []
    for topic in topic_info[topic_info['Topic'] != -1]['Topic']:
        # Get topic words with their scores
        topic_words = topic_model.get_topic(topic)
        if topic_words:
            # Calculate average score of top words as a simple coherence measure
            avg_score = sum(score for _, score in topic_words) / len(topic_words)
            coherence_values.append(avg_score)

    if coherence_values:
        avg_coherence = sum(coherence_values) / len(coherence_values)
        print(f"Average topic coherence (word scores): {avg_coherence:.4f}")
    else:
        print("Could not calculate coherence - no coherence values found")
except Exception as e:
    print(f"Could not calculate coherence: {str(e)}")

# Qualitative evaluation - print top topics
print("\nTop 5 largest topics:")
for idx, row in topic_info.sort_values("Count", ascending=False).head(5).iterrows():
    topic = row["Topic"]
    if topic != -1:  # Skip outlier topic
        words = ", ".join([word for word, _ in topic_model.get_topic(topic)])
        print(f"Topic {topic} (Count: {row['Count']}): {words}")

# Print topic distribution - how evenly documents are distributed
topic_counts = topic_info[topic_info['Topic'] != -1]['Count']
if len(topic_counts) > 0:
    min_count = topic_counts.min()
    max_count = topic_counts.max()
    print(f"Topic size distribution: min={min_count}, max={max_count}, ratio={min_count/max_count:.4f}")